# Chapter 31 — Making Deep Networks Train

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch31/_lib.py`.

In [2]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X, y = digits.data / 16.0, digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                      stratify=y, random_state=0)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def naive_init(sizes, seed):
    r = np.random.default_rng(seed)
    return [r.normal(0, 1, (sizes[i], sizes[i+1])) for i in range(len(sizes)-1)]

def he_init(sizes, seed):
    r = np.random.default_rng(seed)
    return [r.normal(0, np.sqrt(2 / sizes[i]), (sizes[i], sizes[i+1]))
            for i in range(len(sizes) - 1)]

deep_sizes = [64, 64, 64, 64, 64, 10]      # five weight layers, shared by every demo

## The chapter code

### Block 1  (`c1.py`)

In [3]:
# Stack many layers with the naive initialization every tutorial starts
# with: weights drawn from a standard normal. Watch what happens to the
# signal as it passes through, before any training at all.
sizes = [64] + [64] * 8                 # eight layers, same width throughout
Ws = naive_init(sizes, seed=31)
a = X[:200]                              # 200 digits, forward pass only

print(f"{'layer':>7}{'mean |activation|':>20}{'fraction alive':>16}")
print(f"{'input':>7}{np.abs(a).mean():>20.4f}{'--':>16}")
for i, W in enumerate(Ws, 1):
    a = np.maximum(0, a @ W)             # ReLU at every layer
    alive = (a > 0).mean()
    print(f"{i:>7}{np.abs(a).mean():>20,.1f}{alive:>16.4f}")

  layer   mean |activation|  fraction alive
  input              0.3039              --
      1                 1.8          0.5519
      2                 7.5          0.4585
      3                51.1          0.5363
      4               217.6          0.4732
      5             1,112.0          0.5012
      6             6,585.7          0.4784
      7            39,730.9          0.5072
      8           271,077.1          0.5952


### Block 2  (`c2.py`)

In [4]:
# He initialization scales each layer's weights by the width it is
# fanning out from, so variance neither grows nor shrinks layer to layer.
Ws_he = he_init(sizes, seed=31)
a = X[:200]

print(f"{'layer':>7}{'mean |activation|':>20}{'fraction alive':>16}")
print(f"{'input':>7}{np.abs(a).mean():>20.4f}{'--':>16}")
for i, W in enumerate(Ws_he, 1):
    a = np.maximum(0, a @ W)
    alive = (a > 0).mean()
    print(f"{i:>7}{np.abs(a).mean():>20.4f}{alive:>16.4f}")

print(f"\nwhy sqrt(2/fan_in): each output is a sum of fan_in terms, each")
print(f"roughly variance 1 * weight_variance. Setting weight_variance to")
print(f"2/fan_in keeps the sum's variance near 1, and the factor of 2")
print(f"accounts for ReLU discarding half the signal on average.")

  layer   mean |activation|  fraction alive
  input              0.3039              --
      1              0.3130          0.5519
      2              0.2349          0.4585
      3              0.2821          0.5363
      4              0.2125          0.4732
      5              0.1920          0.5012
      6              0.2010          0.4784
      7              0.2143          0.5072
      8              0.2585          0.5952

why sqrt(2/fan_in): each output is a sum of fan_in terms, each
roughly variance 1 * weight_variance. Setting weight_variance to
2/fan_in keeps the sum's variance near 1, and the factor of 2
accounts for ReLU discarding half the signal on average.


### Block 3  (`c3.py`)

In [5]:
# Sigmoid saturates: its gradient is near zero almost everywhere except
# a narrow band around zero. Stack layers of it and the gradient reaching
# the earliest layers vanishes, however good the initialization is.
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

Ws_s = he_init(sizes, seed=31)
a = X[:200]
zs = []
for W in Ws_s:
    z = a @ W
    zs.append(z)
    a = sigmoid(z)

print(f"{'layer':>7}{'max sigmoid grad':>18}{'slopes only':>13}"
      f"{'through weights':>17}")
signal = 1.0
g = np.ones_like(zs[-1])            # a gradient of 1.0 at every output unit
for i, (z, W) in enumerate(zip(reversed(zs), reversed(Ws_s)), 1):
    gmax = sigmoid_grad(z).max()
    signal *= gmax
    g = (g * sigmoid_grad(z)) @ W.T     # the real backward pass, weights too
    print(f"{i:>7}{gmax:>18.4f}{signal:>13.2e}{np.abs(g).mean():>17.2e}")
print(f"\nafter {len(zs)} layers, the slopes alone shrink a gradient by")
print(f"{signal:.2e}; through this network's weights it arrives at")
print(f"{np.abs(g).mean():.2e} of its size at the output.")

  layer  max sigmoid grad  slopes only  through weights
      1            0.2500     2.50e-01         2.38e-01
      2            0.2500     6.25e-02         7.52e-02
      3            0.2500     1.56e-02         2.58e-02
      4            0.2500     3.91e-03         7.00e-03
      5            0.2500     9.76e-04         1.92e-03
      6            0.2500     2.44e-04         6.39e-04
      7            0.2500     6.10e-05         1.81e-04
      8            0.2500     1.53e-05         5.28e-05

after 8 layers, the slopes alone shrink a gradient by
1.53e-05; through this network's weights it arrives at
5.28e-05 of its size at the output.


### Block 4  (`c4.py`)

In [6]:
# Train a genuinely deep network two ways: naive initialization against
# He initialization. Same architecture, same data, same learning rate.
def forward_deep(x, Ws, bs):
    acts = [x]
    for i, (W, b) in enumerate(zip(Ws, bs)):
        z = acts[-1] @ W + b
        a = softmax(z) if i == len(Ws) - 1 else np.maximum(0, z)
        acts.append(a)
    return acts

def train(sizes, init_fn, seed, eta, epochs=250):
    Ws = init_fn(sizes, seed=seed)
    bs = [np.zeros(s) for s in sizes[1:]]
    Ytr = np.eye(10)[ytr]
    history = []
    for epoch in range(epochs + 1):
        acts = forward_deep(Xtr, Ws, bs)
        p = acts[-1]
        loss = -np.sum(Ytr * np.log(p + 1e-12)) / len(Xtr)
        d = (p - Ytr) / len(Xtr)
        grads_W, grads_b = [], []
        for i in reversed(range(len(Ws))):
            gW = acts[i].T @ d
            gb = d.sum(0)
            grads_W.insert(0, gW); grads_b.insert(0, gb)
            if i > 0:
                d = (d @ Ws[i].T) * (acts[i] > 0)
        for i in range(len(Ws)):
            Ws[i] -= eta * grads_W[i]; bs[i] -= eta * grads_b[i]
        if epoch % 50 == 0:
            test_acts = forward_deep(Xte, Ws, bs)
            acc = (test_acts[-1].argmax(1) == yte).mean()
            history.append((epoch, loss, acc))
    return history

print("naive initialization")
for ep, loss, acc in train(deep_sizes, naive_init, seed=31, eta=0.05):
    print(f"  epoch {ep:>4}  loss {loss:>10.4f}  test acc {acc:.4f}")

print("\nHe initialization, same architecture, same learning rate")
for ep, loss, acc in train(deep_sizes, he_init, seed=31, eta=0.05):
    print(f"  epoch {ep:>4}  loss {loss:>10.4f}  test acc {acc:.4f}")

naive initialization


  epoch    0  loss    24.9888  test acc 0.1000
  epoch   50  loss     2.3028  test acc 0.1028
  epoch  100  loss     2.3027  test acc 0.1028
  epoch  150  loss     2.3026  test acc 0.1028
  epoch  200  loss     2.3026  test acc 0.1028
  epoch  250  loss     2.3025  test acc 0.1028

He initialization, same architecture, same learning rate


  epoch    0  loss     2.4655  test acc 0.0972
  epoch   50  loss     0.8202  test acc 0.8194
  epoch  100  loss     0.3258  test acc 0.8889
  epoch  150  loss     0.1886  test acc 0.9167
  epoch  200  loss     0.1278  test acc 0.9444
  epoch  250  loss     0.0999  test acc 0.9583


### Block 5  (`c5.py`)

In [7]:
# Batch normalization rescales each layer's output to zero mean, unit
# variance, using the statistics of the current batch, then lets the
# network learn a scale and shift back if it wants one.
def forward_bn(x, Ws, bs, gammas, betas, eps=1e-5):
    acts = [x]
    for i, (W, b) in enumerate(zip(Ws, bs)):
        z = acts[-1] @ W + b
        if i < len(Ws) - 1:
            mu, var = z.mean(0), z.var(0)
            z_norm = (z - mu) / np.sqrt(var + eps)
            z = gammas[i] * z_norm + betas[i]
            a = np.maximum(0, z)
        else:
            a = softmax(z)
        acts.append(a)
    return acts

def train_bn(sizes, init_fn, seed, eta, epochs=250):
    Ws = init_fn(sizes, seed=seed)
    bs = [np.zeros(s) for s in sizes[1:]]
    gammas = [np.ones(s) for s in sizes[1:-1]]
    betas = [np.zeros(s) for s in sizes[1:-1]]
    Ytr = np.eye(10)[ytr]
    history = []
    for epoch in range(epochs + 1):
        acts = forward_bn(Xtr, Ws, bs, gammas, betas)
        p = acts[-1]
        loss = -np.sum(Ytr * np.log(p + 1e-12)) / len(Xtr)
        d = (p - Ytr) / len(Xtr)
        grads_W, grads_b = [], []
        for i in reversed(range(len(Ws))):
            gW = acts[i].T @ d
            gb = d.sum(0)
            grads_W.insert(0, gW); grads_b.insert(0, gb)
            if i > 0:
                d = (d @ Ws[i].T) * (acts[i] > 0)
        for i in range(len(Ws)):
            Ws[i] -= eta * grads_W[i]; bs[i] -= eta * grads_b[i]
        if epoch % 50 == 0:
            test_acts = forward_bn(Xte, Ws, bs, gammas, betas)
            acc = (test_acts[-1].argmax(1) == yte).mean()
            history.append((epoch, loss, acc))
    return history

print("naive initialization, WITH batch normalization")
for ep, loss, acc in train_bn(deep_sizes, naive_init, seed=31, eta=0.05):
    print(f"  epoch {ep:>4}  loss {loss:>10.4f}  test acc {acc:.4f}")

naive initialization, WITH batch normalization


  epoch    0  loss     8.9811  test acc 0.3083
  epoch   50  loss     1.6946  test acc 0.5833
  epoch  100  loss     1.5755  test acc 0.6194
  epoch  150  loss     1.1951  test acc 0.6389
  epoch  200  loss     0.9892  test acc 0.6500
  epoch  250  loss     0.8389  test acc 0.6806


### Block 6  (`c6.py`)

In [8]:
# The BN implementation above never updates gamma and beta: computing
# their gradients requires differentiating through the normalization
# itself, since the mean and variance both depend on every example in
# the batch. That is real, non-trivial calculus, and it is exactly the
# work automatic differentiation exists to do for you.
#
# What can be checked without it: does the fix from Step 5 depend on
# which naive initialization was unlucky, or is it reliable?
print(f"{'seed':>6}{'no BN, final acc':>18}{'with BN, final acc':>20}")
for seed in (10, 11, 12, 13):
    h1 = train(deep_sizes, naive_init, seed=seed, eta=0.05, epochs=250)
    h2 = train_bn(deep_sizes, naive_init, seed=seed, eta=0.05, epochs=250)
    print(f"{seed:>6}{h1[-1][2]:>18.4f}{h2[-1][2]:>20.4f}")

  seed  no BN, final acc  with BN, final acc


    10            0.1000              0.6583


    11            0.1000              0.7583


    12            0.1028              0.5444


    13            0.1028              0.7278
